# TA City — Phase 0 QC: BehaviorSpace batch checks (R125–R128)

Validates the seed-controlled calibration batch from the BehaviorSpace table:

1. **RED invariants** (MED/HED) must be exactly 0 in every run (R108 check).
2. **Quantity residual** (household-number − new-urban) within tolerance
   (default 0–20 cells; > ~0.1 % of demand = bookkeeping bug, not noise).
3. **Reconciliation**: residual must equal the stranded developers' pre-drawn
   estate sizes in every run.
4. **Demand parity**: every scenario reaches the frozen demand (within the residual).

Run cells top to bottom, or *Restart & Run All*. Reusable for future batches
(including Morris) by editing CONFIG only.

Stdlib only — no numpy/pandas needed.

In [1]:
# CONFIG ---------------------------------------------------------------
CONFIG = {
    "table_path": "../docs/JAK_TA-model_BS_table_R125-128.csv",  # repo-relative or absolute
    "demand": 29656,          # frozen household-number (gross target)
    "final_step": "auto",     # "auto" = highest [step] present, or a number
    "max_residual": 20,       # tolerance for the quantity residual
    "qc_out": None,           # output path; None = next to the table, _BS_QC_
}


In [2]:
# reporter column names in the BehaviorSpace table ------------------------
INV_MED = ("(sum [estate-size] of developers with [class = 2 and "
           "MED-movement = 1]) - count patches with [pMED-movement = 1]")
INV_HED = ("(sum [estate-size] of developers with [class = 3 and "
           "HED-movement = 1]) - count patches with [pHED-movement = 1]")
NEW_URBAN = "(count patches with [new-urban = 1])"
RESIDUAL = "(household-number - count patches with [new-urban = 1])"
STRANDED = ("sum [estate-size] of developers with "
            "[(class = 2 and MED-movement = 0) or (class = 3 and HED-movement = 0)]")

def read_bs_table(path):
    """Read a BehaviorSpace table CSV: 6 preamble lines, then header + rows."""
    from pathlib import Path
    import csv
    path = Path(path)
    lines = path.read_text(encoding="utf-8").splitlines()
    header = [h.strip() for h in next(csv.reader([lines[6]]))]
    rows = [dict(zip(header, r))
            for r in csv.reader(lines[7:])
            if r and len(r) == len(header)]
    return rows

In [3]:
# load table and isolate the final-step rows -----------------------------
rows = read_bs_table(CONFIG["table_path"])
steps = sorted({int(r["[step]"]) for r in rows})
final_step = max(steps) if CONFIG["final_step"] == "auto" else int(CONFIG["final_step"])
final = [r for r in rows if int(r["[step]"]) == final_step]
assert final, f"no rows at final step {final_step}"
print(f"table: {CONFIG['table_path']}")
print(f"steps present: {min(steps)}..{max(steps)} | final-step rows: {len(final)}")
print(f"final step used: {final_step}")

table: ../docs/JAK_TA-model_BS_table_R125-128.csv
steps present: 0..25 | final-step rows: 12
final step used: 25


In [4]:
# CHECK 1-2: RED invariants + quantity residual -------------------------
ok = True
bad = [r for r in final if int(r[INV_MED]) != 0 or int(r[INV_HED]) != 0]
if bad:
    ok = False
    for r in bad:
        print(f"FAIL RED invariant: run {r['[run number]']} {r['scenario']} seed {r['run-seed']}")
else:
    print("PASS  RED invariants (MED & HED) = 0 in all final-step runs")

residuals = sorted({int(r[RESIDUAL]) for r in final})
if residuals[0] < 0 or residuals[-1] > CONFIG["max_residual"]:
    ok = False
    print(f"FAIL quantity residual outside [0, {CONFIG['max_residual']}]: {residuals}")
else:
    print(f"PASS  quantity residual in [0, {CONFIG['max_residual']}]: min={residuals[0]} max={residuals[-1]}")

PASS  RED invariants (MED & HED) = 0 in all final-step runs
PASS  quantity residual in [0, 20]: min=0 max=9


In [5]:
# CHECK 3: residual reconciles with stranded developers (R108 invariant) -
for r in sorted(final, key=lambda x: (x["scenario"], int(x["run-seed"]))):
    if int(r[RESIDUAL]) != int(r[STRANDED]):
        ok = False
        print(f"FAIL reconciliation: run {r['[run number]']} {r['scenario']} "
              f"residual={r[RESIDUAL]} stranded-estate={r[STRANDED]} (must be equal)")
else:
    print("PASS  residual == stranded-estate sizes in every run"
          if ok else "see failures above")

PASS  residual == stranded-estate sizes in every run


In [6]:
# CHECK 4: final cells per run + demand parity ---------------------------
from collections import defaultdict
by_scen = defaultdict(list)
for r in final:
    by_scen[r["scenario"]].append(int(r[NEW_URBAN]))

print("run seed scenario      new-urban  resid")
for r in sorted(final, key=lambda x: (x["scenario"], int(x["run-seed"]))):
    print(f"{r['[run number]']:>3} {r['run-seed']:>4} {r['scenario']:<12} "
          f"{r[NEW_URBAN]:>9} {r[RESIDUAL]:>6}")

print(f"\nfinal new-urban by scenario (min / mean / max, parity vs demand {CONFIG['demand']}):")
for s in sorted(by_scen):
    v = by_scen[s]
    drift = (max(max(v), CONFIG["demand"]) - min(min(v), CONFIG["demand"])) / CONFIG["demand"]
    flag = "PASS" if drift <= 0.001 else "WARN"
    print(f"  {s:<12} {min(v):>6} / {sum(v)/len(v):>9.1f} / {max(v):>6}  "
          f"spread vs demand: {drift*100:.2f}% [{flag}]")

run seed scenario      new-urban  resid
  1   11 s0               29647      9
  5   22 s0               29647      9
  9   33 s0               29647      9
  3   11 s_edge           29650      6
  7   22 s_edge           29653      3
 11   33 s_edge           29656      0
  2   11 s_random         29656      0
  6   22 s_random         29650      6
 10   33 s_random         29653      3
  4   11 s_roadbound      29656      0
  8   22 s_roadbound      29656      0
 12   33 s_roadbound      29656      0

final new-urban by scenario (min / mean / max, parity vs demand 29656):
  s0            29647 /   29647.0 /  29647  spread vs demand: 0.03% [PASS]
  s_edge        29650 /   29653.0 /  29656  spread vs demand: 0.02% [PASS]
  s_random      29650 /   29653.0 /  29656  spread vs demand: 0.02% [PASS]
  s_roadbound   29656 /   29656.0 /  29656  spread vs demand: 0.00% [PASS]


In [7]:
# write QC table --------------------------------------------------------
from pathlib import Path
import csv
table_path = Path(CONFIG["table_path"])
out = Path(CONFIG["qc_out"]) if CONFIG["qc_out"] else table_path.parent / table_path.name.replace(
    "BS_table", "BS_QC")
fields = ["[run number]", "run-seed", "scenario", NEW_URBAN, RESIDUAL,
          INV_MED, INV_HED, STRANDED, "[step]"]
with out.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    for r in sorted(final, key=lambda x: (x["scenario"], int(x["run-seed"]))):
        w.writerow(r)
print(f"QC table written: {out}")
print("OVERALL:", "PASS" if ok else "FAIL - investigate flagged runs")

QC table written: ..\docs\JAK_TA-model_BS_QC_R125-128.csv
OVERALL: PASS
